<a href="https://colab.research.google.com/github/Akash14-09/Kaggle-/blob/main/Logistic_Reg_on_Rain_in_Australia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Logistic Regression Implementation on Rain in Australia Dataset

###Dataset Link - https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package/data

In [33]:
import numpy as np
import pandas as pd

In [34]:
df = pd.read_csv('/content/weatherAUS.csv')
df

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145455,2017-06-21,Uluru,2.8,23.4,0.0,NaN,NaN,E,31.0,SE,...,51.0,24.0,1024.6,1020.3,NaN,NaN,10.1,22.4,No,No
145456,2017-06-22,Uluru,3.6,25.3,0.0,NaN,NaN,NNW,22.0,SE,...,56.0,21.0,1023.5,1019.1,NaN,NaN,10.9,24.5,No,No
145457,2017-06-23,Uluru,5.4,26.9,0.0,NaN,NaN,N,37.0,SE,...,53.0,24.0,1021.0,1016.8,NaN,NaN,12.5,26.1,No,No
145458,2017-06-24,Uluru,7.8,27.0,0.0,NaN,NaN,SE,28.0,SSE,...,51.0,24.0,1019.4,1016.5,3.0,2.0,15.1,26.0,No,No


In [35]:
missing = df.isnull().sum()
missing_percentage = (missing/len(df))*100

summary = pd.DataFrame({'Missing_count': missing, 'Percentage': missing_percentage})

print(summary[summary['Missing_count']>0].sort_values('Percentage', ascending=False))
print()

               Missing_count  Percentage
Sunshine               69835   48.009762
Evaporation            62790   43.166506
Cloud3pm               59358   40.807095
Cloud9am               55888   38.421559
Pressure9am            15065   10.356799
Pressure3pm            15028   10.331363
WindDir9am             10566    7.263853
WindGustDir            10326    7.098859
WindGustSpeed          10263    7.055548
Humidity3pm             4507    3.098446
WindDir3pm              4228    2.906641
Temp3pm                 3609    2.481094
RainTomorrow            3267    2.245978
Rainfall                3261    2.241853
RainToday               3261    2.241853
WindSpeed3pm            3062    2.105046
Humidity9am             2654    1.824557
WindSpeed9am            1767    1.214767
Temp9am                 1767    1.214767
MinTemp                 1485    1.020899
MaxTemp                 1261    0.866905



In [36]:
df.shape
#here we have 145460 rows and 23 columns

(145460, 23)

#Handling the Missing Values

1.Sunshine,Evaporation,Cloud3pm , Cloud9am are the columns which have more than 10% missing values so we are going to drop them.


2.So what do we look at instead?

Precision (for the "Yes" class): Of all the days the model predicted rain, how many actually had rain? High precision = few false alarms.

Recall: Of all the days it actually rained, how many did the model catch? High recall = few missed rainy days.

F1-score: A balance between precision and recall in one number.

ROC-AUC: Measures how well the model separates the two classes overall, regardless of imbalance.



In [37]:

# Step 1: Drop sparse columns
df = df.drop(columns=['Sunshine', 'Evaporation', 'Cloud9am', 'Cloud3pm'])

# Step 2: Drop rows with missing target
df = df.dropna(subset=['RainTomorrow'])

# Step 3: Extract Month from Date, drop Date
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df = df.drop(columns=['Date'])

# Step 4: Encode RainToday/RainTomorrow as 0/1
df['RainToday'] = df['RainToday'].map({'Yes': 1, 'No': 0})
df['RainTomorrow'] = df['RainTomorrow'].map({'Yes': 1, 'No': 0})


In [38]:
df.shape
#now these are the remaining rows and coulmns that we have

(142193, 19)

In [39]:
#imputing missing values
#numeric --> median
#categorical --> mode
#then applying one hot encoding on the categorical columns

#imputing missing values
#numeric --> median
#categorical --> mode
#then applying one hot encoding on the categorical columns
 #(and becuase of one hot encoding we would be having 108 columns instead of 19 because for each will create (n-1) columns )




# Identify column types
categorical_cols = ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']
# RainToday is binary numeric-ish but has NaNs - impute with mode
binary_cols = ['RainToday']
numeric_cols = [c for c in df.columns if c not in categorical_cols + binary_cols + ['RainTomorrow']]

# Impute numeric columns with median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())


# Impute categorical + RainToday with mode
for col in categorical_cols + binary_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print('Missing values after imputation:')
print(df.isnull().sum().sum(), 'total missing values remaining')
print()
print('Numeric columns imputed:', numeric_cols)
print('Categorical columns imputed:', categorical_cols)
print()

# One-hot encode categorical columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print('Shape after one-hot encoding:', df_encoded.shape)
print()
print('Sample of new columns created:', [c for c in df_encoded.columns if 'Location_' in c][:5], '...')
#now the new columns has been created with the help of one hot encoding



Missing values after imputation:
0 total missing values remaining

Numeric columns imputed: ['MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm', 'Month']
Categorical columns imputed: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']

Shape after one-hot encoding: (142193, 108)

Sample of new columns created: ['Location_Albany', 'Location_Albury', 'Location_AliceSprings', 'Location_BadgerysCreek', 'Location_Ballarat'] ...


In [40]:
categorical_cols = ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

####Training and Testing

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score



X = df.drop(columns = ['RainTomorrow'])
y = df['RainTomorrow']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42 , stratify = y)

In [42]:
print('Train Shape: ' , X_train.shape)
print('Test Shape: ' , X_test.shape)

print()
print()
print('Train target distribution : ')
print()
print()
print(y_train.value_counts(normalize=True).round(4))
print()
print()
print('Test target distribution : ')
print()
print()
print(y_test.value_counts(normalize=True).round(4))


#it means both the testing and the training data has equal distribution of yes and no




# Scale only the original numeric columns (not one-hot dummies, not RainToday which is binary)
numeric_cols = ['MinTemp','MaxTemp','Rainfall','WindGustSpeed','WindSpeed9am','WindSpeed3pm',
                'Humidity9am','Humidity3pm','Pressure9am','Pressure3pm','Temp9am','Temp3pm','Month']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

#Scaling the new dataset

Train Shape:  (113754, 107)
Test Shape:  (28439, 107)


Train target distribution : 


RainTomorrow
0    0.7758
1    0.2242
Name: proportion, dtype: float64


Test target distribution : 


RainTomorrow
0    0.7758
1    0.2242
Name: proportion, dtype: float64


In [43]:
df

,MinTemp,MaxTemp,Rainfall,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,...,WindDir3pm_NNW,WindDir3pm_NW,WindDir3pm_S,WindDir3pm_SE,WindDir3pm_SSE,WindDir3pm_SSW,WindDir3pm_SW,WindDir3pm_W,WindDir3pm_WNW,WindDir3pm_WSW
0,13.4,22.9,0.6,44.0,20.0,24.0,71.0,22.0,1007.7,1007.1,...,False,False,False,False,False,False,False,False,True,False
1,7.4,25.1,0.0,44.0,4.0,22.0,44.0,25.0,1010.6,1007.8,...,False,False,False,False,False,False,False,False,False,True
2,12.9,25.7,0.0,46.0,19.0,26.0,38.0,30.0,1007.6,1008.7,...,False,False,False,False,False,False,False,False,False,True
3,9.2,28.0,0.0,24.0,11.0,9.0,45.0,16.0,1017.6,1012.8,...,False,False,False,False,False,False,False,False,False,False
4,17.5,32.3,1.0,41.0,7.0,20.0,82.0,33.0,1010.8,1006.0,...,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145454,3.5,21.8,0.0,31.0,15.0,13.0,59.0,27.0,1024.7,1021.2,...,False,False,False,False,False,False,False,False,False,False
145455,2.8,23.4,0.0,31.0,13.0,11.0,51.0,24.0,1024.6,1020.3,...,False,False,False,False,False,False,False,False,False,False
145456,3.6,25.3,0.0,22.0,13.0,9.0,56.0,21.0,1023.5,1019.1,...,False,False,False,False,False,False,False,False,False,False
145457,5.4,26.9,0.0,37.0,9.0,9.0,53.0,24.0,1021.0,1016.8,...,False,False,False,False,False,False,False,False,True,False


###Implementing Logistic Regression

In [52]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score , precision_score , recall_score , f1_score , roc_auc_score)


model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]




/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


###Results

In [53]:

print('Final Evaluation')
print()
print()



print('Accuracy: ', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred), 4))
print('Recall:   ', round(recall_score(y_test, y_pred), 4))
print('F1 Score: ', round(f1_score(y_test, y_pred), 4))
print('ROC-AUC:  ', round(roc_auc_score(y_test, y_proba), 4))


Final Evaluation


Accuracy:  0.7843
Precision: 0.5128
Recall:    0.7589
F1 Score:  0.612
ROC-AUC:   0.8575
